### ***Phase 1***
- Data Loading & Schema Handling

In [ ]:
%pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("COVID19-Analytics-Pipeline").getOrCreate()

In [4]:
# Load all datasets

full_grouped = spark.read.csv("datasets/full_grouped.csv", header=True, inferSchema=True)
covid_clean = spark.read.csv("datasets/covid_19_clean_complete.csv", header=True, inferSchema=True)
country_latest = spark.read.csv("datasets/country_wise_latest.csv", header=True, inferSchema=True)
day_wise = spark.read.csv("datasets/day_wise.csv", header=True, inferSchema=True)
usa_county = spark.read.csv("datasets/usa_county_wise.csv", header=True, inferSchema=True)
worldometer = spark.read.csv("datasets/worldometer_data.csv", header=True, inferSchema=True)

In [ ]:
# Schema of each dataset

print("FULL GROUPED SCHEMA")
full_grouped.printSchema()

print("COVID CLEAN SCHEMA")
covid_clean.printSchema()

print("COUNTRY LATEST SCHEMA")
country_latest.printSchema()

print("DAY WISE SCHEMA")
day_wise.printSchema()

print("USA COUNTY SCHEMA")
usa_county.printSchema()

print("WORLDMETER SCHEMA")
worldometer.printSchema()

FULL GROUPED SCHEMA
root
 |-- Date: date (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- Confirmed: integer (nullable = true)
 |-- Deaths: integer (nullable = true)
 |-- Recovered: integer (nullable = true)
 |-- Active: integer (nullable = true)
 |-- New cases: integer (nullable = true)
 |-- New deaths: integer (nullable = true)
 |-- New recovered: integer (nullable = true)
 |-- WHO Region: string (nullable = true)

COVID CLEAN SCHEMA
root
 |-- Province/State: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- Lat: double (nullable = true)
 |-- Long: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Confirmed: integer (nullable = true)
 |-- Deaths: integer (nullable = true)
 |-- Recovered: integer (nullable = true)
 |-- Active: integer (nullable = true)
 |-- WHO Region: string (nullable = true)

COUNTRY LATEST SCHEMA
root
 |-- Country/Region: string (nullable = true)
 |-- Confirmed: integer (nullable = true)
 |-- Deaths: integ

In [6]:
# Count Rows in Each dataset

print("Full Grouped Rows:", full_grouped.count())
print("Covid Clean Rows:", covid_clean.count())
print("Country Latest Rows:", country_latest.count())
print("Day Wise Rows:", day_wise.count())
print("USA County Rows:", usa_county.count())
print("Worldometer Rows:", worldometer.count())

Full Grouped Rows: 35156
Covid Clean Rows: 49068
Country Latest Rows: 187
Day Wise Rows: 188
USA County Rows: 627920
Worldometer Rows: 209


### ***Phase 2***
- Data Cleaning

#### Task 2 : Handle Missing Province/State Values


In [ ]:
from pyspark.sql.functions import col
covid_clean.filter(col("Province/State").isNull()).show()

+--------------+--------------------+---------+----------+----------+---------+------+---------+------+--------------------+
|Province/State|      Country/Region|      Lat|      Long|      Date|Confirmed|Deaths|Recovered|Active|          WHO Region|
+--------------+--------------------+---------+----------+----------+---------+------+---------+------+--------------------+
|          NULL|         Afghanistan| 33.93911| 67.709953|2020-01-22|        0|     0|        0|     0|Eastern Mediterra...|
|          NULL|             Albania|  41.1533|   20.1683|2020-01-22|        0|     0|        0|     0|              Europe|
|          NULL|             Algeria|  28.0339|    1.6596|2020-01-22|        0|     0|        0|     0|              Africa|
|          NULL|             Andorra|  42.5063|    1.5218|2020-01-22|        0|     0|        0|     0|              Europe|
|          NULL|              Angola| -11.2027|   17.8739|2020-01-22|        0|     0|        0|     0|              Africa|


In [8]:
covid_clean = covid_clean.fillna({"Province/State": "Unknown"})

In [9]:
null_report = covid_clean.filter(col("Province/State") == "Unknown") \
    .groupBy("Country/Region") \
    .count() \
    .orderBy("count", ascending=False)
null_report.show()

+--------------+-----+
|Country/Region|count|
+--------------+-----+
|          Chad|  188|
|      Paraguay|  188|
|        Russia|  188|
|         Yemen|  188|
|       Senegal|  188|
|    Cabo Verde|  188|
|        Sweden|  188|
|        Guyana|  188|
|       Eritrea|  188|
|   Philippines|  188|
|         Burma|  188|
|      Djibouti|  188|
|      Malaysia|  188|
|     Singapore|  188|
|          Fiji|  188|
|        Turkey|  188|
|        Malawi|  188|
|Western Sahara|  188|
|          Iraq|  188|
|       Germany|  188|
+--------------+-----+
only showing top 20 rows


#### Task 3 : Standardize Country Names

In [ ]:
from pyspark.sql.functions import when

def standardize_country(df):
    return df.withColumn(
        "Country/Region",
        when(col("Country/Region") == "US", "USA")
        .when(col("Country/Region") == "Korea, South", "South Korea")
        .when(col("Country/Region") == "UK", "United Kingdom")
        .otherwise(col("Country/Region"))
    )

covid_clean = standardize_country(covid_clean)
full_grouped = standardize_country(full_grouped)
country_latest = standardize_country(country_latest)
worldometer = standardize_country(worldometer)



In [13]:
covid_clean.select("Country/Region").distinct().show(50, False)

+-----------------+
|Country/Region   |
+-----------------+
|Chad             |
|Paraguay         |
|Russia           |
|Yemen            |
|Senegal          |
|Cabo Verde       |
|Sweden           |
|Guyana           |
|Eritrea          |
|Philippines      |
|Burma            |
|Djibouti         |
|Malaysia         |
|Singapore        |
|Fiji             |
|Turkey           |
|Malawi           |
|Western Sahara   |
|Iraq             |
|Germany          |
|Comoros          |
|Afghanistan      |
|Cambodia         |
|Jordan           |
|Maldives         |
|Rwanda           |
|Sudan            |
|France           |
|Greece           |
|Kosovo           |
|Sri Lanka        |
|Dominica         |
|Algeria          |
|Equatorial Guinea|
|Togo             |
|Slovakia         |
|Argentina        |
|Angola           |
|Belgium          |
|San Marino       |
|Ecuador          |
|Qatar            |
|Lesotho          |
|Albania          |
|Madagascar       |
|Finland          |
|Ghana            |


#### Task 4: Remove Duplicate Daily Records

In [ ]:
covid_clean.groupBy("Country/Region", "Date") \
    .count() \
    .filter("count > 1") \
    .show()

+--------------+----------+-----+
|Country/Region|      Date|count|
+--------------+----------+-----+
|         China|2020-05-01|   33|
|   Netherlands|2020-06-08|    4|
|     Australia|2020-06-10|    8|
|       Denmark|2020-06-21|    2|
|United Kingdom|2020-07-03|   11|
|   Netherlands|2020-04-20|    4|
|     Australia|2020-05-10|    8|
|     Australia|2020-07-16|    8|
|United Kingdom|2020-05-03|   11|
|     Australia|2020-05-14|    8|
|     Australia|2020-05-29|    8|
|         China|2020-06-06|   33|
|   Netherlands|2020-07-12|    4|
|        France|2020-07-13|   11|
|        Canada|2020-07-25|   12|
|       Denmark|2020-07-25|    2|
|         China|2020-01-28|   33|
|       Denmark|2020-02-25|    2|
|     Australia|2020-05-01|    8|
|        France|2020-05-27|   11|
+--------------+----------+-----+
only showing top 20 rows


In [16]:
covid_clean = covid_clean.dropDuplicates(["Country/Region", "Date"])
print("After deduplication:", covid_clean.count())

After deduplication: 35156
